# Syniscopy SAM2 inference quick start

Use this notebook after `sam2training.ipynb` has produced a checkpoint.

1. Edit `MICROSCOPE_LABEL` in the setup cell so it matches the training run.
2. Run the setup/model cells. The notebook loads `MyDrive/Syniscopy/<MICROSCOPE_LABEL>/weights/final_checkpoint.pt` and the checkpoint manifest written by training.
3. Upload a video directly into the Colab runtime when prompted.
4. Use the interactive prompt cells to place point and optional box prompts.
5. Output masks and videos are written to `MyDrive/Syniscopy/<MICROSCOPE_LABEL>/inference_outputs/`.

The inference notebook uses the same SAM2.1 Large base checkpoint/config as the training notebook.


In [ ]:
#1
!nvidia-smi

In [ ]:
#2: Mount Drive and select microscope/configuration
import os, re, shutil, json
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')
HOME = '/content'
print('HOME:', HOME)

# ========================== EDIT THIS LINE ==========================
# Must exactly match the label used in sam2training.ipynb.
MICROSCOPE_LABEL = 'default_configuration'
# ======================== END EDIT LINE ========================

RELEASE_NOTEBOOK_VERSION = 'syniscopy-sam2-inference-public-release-2026-05-15'

def sanitize_label(label: str) -> str:
    clean = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(label).strip())
    return clean.strip('._-') or 'default_configuration'

MICROSCOPE_LABEL = sanitize_label(MICROSCOPE_LABEL)
DRIVE_ROOT = Path('/content/drive/MyDrive/Syniscopy')
CONFIG_DIR = DRIVE_ROOT / MICROSCOPE_LABEL
WEIGHTS_DIR = CONFIG_DIR / 'weights'
FINAL_CHECKPOINT_PATH = WEIGHTS_DIR / 'final_checkpoint.pt'
CHECKPOINT_MANIFEST_PATH = WEIGHTS_DIR / 'checkpoint_manifest.json'
INFERENCE_OUTPUT_ROOT = CONFIG_DIR / 'inference_outputs'
INFERENCE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SAM2_CONFIG_NAME = 'configs/sam2.1/sam2.1_hiera_l.yaml'
SAM2_BASE_CHECKPOINT_FILENAME = 'sam2.1_hiera_large.pt'
SAM2_BASE_CHECKPOINT_URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'

if CHECKPOINT_MANIFEST_PATH.exists():
    with open(CHECKPOINT_MANIFEST_PATH, 'r') as f:
        checkpoint_manifest = json.load(f)
    SAM2_CONFIG_NAME = checkpoint_manifest.get('sam2_config_name', SAM2_CONFIG_NAME)
    SAM2_BASE_CHECKPOINT_FILENAME = checkpoint_manifest.get('sam2_base_checkpoint_filename', SAM2_BASE_CHECKPOINT_FILENAME)
    SAM2_BASE_CHECKPOINT_URL = checkpoint_manifest.get('sam2_base_checkpoint_url', SAM2_BASE_CHECKPOINT_URL)
    print('Loaded checkpoint manifest:', CHECKPOINT_MANIFEST_PATH)
else:
    checkpoint_manifest = None
    print('No checkpoint manifest found; using SAM2.1 Large defaults.')

if not FINAL_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f'No final checkpoint found for label {MICROSCOPE_LABEL!r}: {FINAL_CHECKPOINT_PATH}. '
        'Run the training notebook for this label first.'
    )

print('Notebook version:', RELEASE_NOTEBOOK_VERSION)
print('Microscope/config label:', MICROSCOPE_LABEL)
print('Using fine-tuned checkpoint:', FINAL_CHECKPOINT_PATH)
print('SAM2 config:', SAM2_CONFIG_NAME)
print('SAM2 base checkpoint:', SAM2_BASE_CHECKPOINT_FILENAME)


In [ ]:
#3: Clone/install SAM2
import os
import shutil
import subprocess
import sys
from pathlib import Path

SAM2_DIR = Path(HOME) / 'segment-anything-2'
SAM2_GIT_URL = 'https://github.com/facebookresearch/segment-anything-2.git'
SAM2_GIT_COMMIT = '2b90b9f5ceec907a1c18123530e92e794ad901a4'
os.chdir(HOME)


def _current_git_commit(repo_dir: Path) -> str | None:
    try:
        out = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'],
            cwd=str(repo_dir),
            stderr=subprocess.DEVNULL,
            text=True,
        )
    except Exception:
        return None
    return out.strip() or None


repo_valid = (
    (SAM2_DIR / 'setup.py').exists()
    and (SAM2_DIR / 'sam2').is_dir()
    and _current_git_commit(SAM2_DIR) == SAM2_GIT_COMMIT
)
if repo_valid:
    print(f'Reusing pinned SAM2 repository: {SAM2_DIR} @ {SAM2_GIT_COMMIT[:12]}')
else:
    if SAM2_DIR.exists():
        print('Removing SAM2 directory that is missing files or not at the pinned commit...')
        shutil.rmtree(SAM2_DIR)
    subprocess.run(['git', 'clone', SAM2_GIT_URL, str(SAM2_DIR)], check=True)
    subprocess.run(['git', 'checkout', '-q', SAM2_GIT_COMMIT], cwd=str(SAM2_DIR), check=True)
    if not (SAM2_DIR / 'setup.py').exists():
        raise RuntimeError('SAM2 clone failed: setup.py missing')
    actual_commit = _current_git_commit(SAM2_DIR)
    if actual_commit != SAM2_GIT_COMMIT:
        raise RuntimeError(f'SAM2 checkout mismatch: {actual_commit} != {SAM2_GIT_COMMIT}')

%cd {SAM2_DIR}
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], check=True)


In [ ]:
#4
!pip install -q supervision[assets] jupyter_bbox_widget

In [ ]:
#5: Verify/download the SAM2.1 Large base checkpoint used by training
from pathlib import Path

SAM2_DIR = Path(HOME) / 'segment-anything-2'
CHECKPOINT = SAM2_DIR / SAM2_BASE_CHECKPOINT_FILENAME

if CHECKPOINT.exists() and CHECKPOINT.stat().st_size > 100 * 1024 * 1024:
    print('Base checkpoint exists:', CHECKPOINT)
else:
    print('Downloading base checkpoint:', SAM2_BASE_CHECKPOINT_URL)
    !wget -q --show-progress -O {CHECKPOINT} {SAM2_BASE_CHECKPOINT_URL}
    if not CHECKPOINT.exists() or CHECKPOINT.stat().st_size < 100 * 1024 * 1024:
        raise RuntimeError(f'Base checkpoint download failed: {CHECKPOINT}')
print('Base checkpoint:', CHECKPOINT)


In [ ]:
#6
import cv2
import torch
import base64

import numpy as np
import supervision as sv

from pathlib import Path
from supervision.assets import download_assets, VideoAssets
from sam2.build_sam import build_sam2_video_predictor

IS_COLAB = True

if IS_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

from jupyter_bbox_widget import BBoxWidget

In [ ]:
# Cell 7: Mixed precision
# Removed mixed precision for potentially higher accuracy at increased GPU load
# torch.autocast(device_type="cuda", dtype=torch.bfloat16).__enter__()

# if torch.cuda.get_device_properties(0).major >= 8:
#     torch.backends.cuda.matmul.allow_tf32 = True
#     torch.backends.cudnn.allow_tf32 = True

In [ ]:
# Cell 8: Prepare base and fine-tuned SAM2 predictors
import gc
import os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = str(Path(HOME) / 'segment-anything-2' / SAM2_BASE_CHECKPOINT_FILENAME)
CONFIG = SAM2_CONFIG_NAME
FINE_TUNED_WEIGHTS = str(FINAL_CHECKPOINT_PATH)

if not os.path.exists(FINE_TUNED_WEIGHTS):
    raise FileNotFoundError(f'Fine-tuned checkpoint not found: {FINE_TUNED_WEIGHTS}')

print('Base SAM2 checkpoint:', CHECKPOINT)
print('Fine-tuned checkpoint:', FINE_TUNED_WEIGHTS)

base_ckpt = torch.load(CHECKPOINT, map_location='cpu')
base_model_state = base_ckpt.get('model', base_ckpt) if isinstance(base_ckpt, dict) else base_ckpt
custom_ckpt = torch.load(FINE_TUNED_WEIGHTS, map_location='cpu')
FINE_TUNED_STATE_DICT = custom_ckpt.get('model', custom_ckpt) if isinstance(custom_ckpt, dict) else custom_ckpt

FINETUNED_COMPATIBLE_KEYS = []
FINETUNED_CHANGED_KEYS = []
FINETUNED_SKIPPED_SHAPE = []
FINETUNED_SKIPPED_MISSING = []
for key, value in FINE_TUNED_STATE_DICT.items():
    clean_key = key.replace('module.', '')
    base_value = base_model_state.get(clean_key)
    if base_value is None:
        FINETUNED_SKIPPED_MISSING.append(clean_key)
        continue
    if tuple(base_value.shape) != tuple(value.shape):
        FINETUNED_SKIPPED_SHAPE.append(clean_key)
        continue
    FINETUNED_COMPATIBLE_KEYS.append(clean_key)
    if not torch.equal(base_value.cpu(), value.cpu()):
        FINETUNED_CHANGED_KEYS.append(clean_key)

if not FINETUNED_COMPATIBLE_KEYS:
    raise RuntimeError('No compatible fine-tuned parameters match the SAM2 base checkpoint.')
if not FINETUNED_CHANGED_KEYS:
    raise RuntimeError('Fine-tuned checkpoint is compatible but does not differ from the base checkpoint.')

print('Checkpoint comparison')
print(f'  Base model keys:              {len(base_model_state)}')
print(f'  Fine-tuned model keys:        {len(FINE_TUNED_STATE_DICT)}')
print(f'  Compatible keys:              {len(FINETUNED_COMPATIBLE_KEYS)}')
print(f'  Compatible keys changed:      {len(FINETUNED_CHANGED_KEYS)}')
print(f'  Skipped shape mismatches:     {len(FINETUNED_SKIPPED_SHAPE)}')
print(f'  Skipped missing in base:      {len(FINETUNED_SKIPPED_MISSING)}')

# Keep only the fine-tuned state dict for later sequential predictor builds.
del base_ckpt, base_model_state, custom_ckpt
gc.collect()

def build_sam2_predictor_variant(model_variant: str):
    """Build one predictor variant. Variants are run sequentially to conserve GPU memory."""
    model_variant = str(model_variant).strip().lower()
    predictor = build_sam2_video_predictor(CONFIG, CHECKPOINT)
    load_report = {
        'model_variant': model_variant,
        'base_checkpoint': CHECKPOINT,
        'fine_tuned_checkpoint': FINE_TUNED_WEIGHTS,
        'loaded_compatible_keys': 0,
        'missing_after_load': 0,
        'unexpected_after_load': 0,
        'skipped_shape': 0,
        'skipped_missing': 0,
    }
    if model_variant == 'base':
        predictor.eval()
        return predictor, load_report
    if model_variant != 'finetuned':
        raise ValueError(f'Unknown model variant: {model_variant!r}')

    current_state = predictor.state_dict()
    compatible_state = {}
    skipped_shape = []
    skipped_missing = []
    for key, value in FINE_TUNED_STATE_DICT.items():
        clean_key = key.replace('module.', '')
        if clean_key in current_state:
            if tuple(current_state[clean_key].shape) == tuple(value.shape):
                compatible_state[clean_key] = value
            else:
                skipped_shape.append(clean_key)
        else:
            skipped_missing.append(clean_key)
    if not compatible_state:
        raise RuntimeError('No compatible fine-tuned parameters matched the SAM2 video predictor.')

    result = predictor.load_state_dict(compatible_state, strict=False)
    predictor.eval()
    load_report.update({
        'loaded_compatible_keys': len(compatible_state),
        'missing_after_load': len(result.missing_keys),
        'unexpected_after_load': len(result.unexpected_keys),
        'skipped_shape': len(skipped_shape),
        'skipped_missing': len(skipped_missing),
    })
    print(f'Loaded fine-tuned predictor: {len(compatible_state)} compatible parameters')
    return predictor, load_report


In [ ]:
# Cell 8.5: Checkpoint summary
print('Checkpoint summary')
print(f'  Base SAM2.1 Large path:       {CHECKPOINT}')
print(f'  Fine-tuned path:              {FINE_TUNED_WEIGHTS}')
print(f'  Compatible fine-tuned keys:   {len(FINETUNED_COMPATIBLE_KEYS)}')
print(f'  Changed compatible keys:      {len(FINETUNED_CHANGED_KEYS)}')
print('  This notebook runs explicit model variants; base and fine-tuned outputs are saved separately when requested.')


In [ ]:
# Cell 9: Upload source video directly into the Colab runtime
import json
import subprocess


def read_video_info(video_path):
    try:
        return sv.VideoInfo.from_video_path(str(video_path))
    except Exception:
        pass
    cap = cv2.VideoCapture(str(video_path))
    if cap.isOpened():
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        if width > 0 and height > 0 and fps > 0 and total > 0:
            return sv.VideoInfo(width=width, height=height, fps=fps, total_frames=total)
    else:
        cap.release()
    try:
        raw = subprocess.check_output(
            [
                'ffprobe', '-v', 'error', '-select_streams', 'v:0',
                '-show_entries', 'stream=width,height,nb_frames,r_frame_rate,duration',
                '-of', 'json', str(video_path),
            ],
            text=True,
        )
        stream = (json.loads(raw).get('streams') or [{}])[0]
        width = int(stream.get('width') or 0)
        height = int(stream.get('height') or 0)
        total = int(stream.get('nb_frames') or 0)
        rate = stream.get('r_frame_rate') or '0/0'
        num, den = rate.split('/')
        fps = float(num) / float(den) if float(den) else 0.0
        if total <= 0 and stream.get('duration') and fps > 0:
            total = int(round(float(stream['duration']) * fps))
        if width > 0 and height > 0 and fps > 0 and total > 0:
            return sv.VideoInfo(width=width, height=height, fps=fps, total_frames=total)
    except Exception as exc:
        print('ffprobe video metadata fallback failed:', repr(exc))
    raise RuntimeError(f'Could not read video metadata for {video_path}')


UPLOAD_DIR = Path('/content/syniscopy_inference_uploads')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No video uploaded.')

uploaded_name = next(iter(uploaded.keys()))
SOURCE_VIDEO = str(UPLOAD_DIR / uploaded_name)
with open(SOURCE_VIDEO, 'wb') as f:
    f.write(uploaded[uploaded_name])

print('Uploaded video:', SOURCE_VIDEO)
SOURCE_VIDEO_INFO = read_video_info(Path(SOURCE_VIDEO))
SOURCE_VIDEO_FPS = float(SOURCE_VIDEO_INFO.fps)
SOURCE_VIDEO_TOTAL_FRAMES = int(SOURCE_VIDEO_INFO.total_frames)
SOURCE_VIDEO_INFO


In [ ]:
# Cell 10: Set extraction parameters
START_IDX = 0
END_IDX = None        # None means through the end of the uploaded video.
FRAME_STRIDE = 1
TARGET_LONG_SIDE = 1024


In [ ]:
# Cell 11: Extract frames for SAM2 inference
import subprocess

SOURCE_FRAMES = Path(HOME) / f"{Path(SOURCE_VIDEO).stem}_frames"
SOURCE_FRAMES.mkdir(parents=True, exist_ok=True)
for old in SOURCE_FRAMES.glob('*.jpeg'):
    old.unlink()

start = max(0, int(START_IDX))
stride = max(1, int(FRAME_STRIDE))
written = 0
cap = cv2.VideoCapture(SOURCE_VIDEO)
if cap.isOpened():
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    end = total if END_IDX is None else min(total, int(END_IDX))
    if start >= end:
        raise ValueError(f'Invalid frame range: START_IDX={START_IDX}, END_IDX={END_IDX}, decoded total={total}')
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    source_idx = start
    while source_idx < end:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        if (source_idx - start) % stride == 0:
            h, w = frame_bgr.shape[:2]
            scale = min(1.0, float(TARGET_LONG_SIDE) / max(h, w)) if TARGET_LONG_SIDE else 1.0
            if scale != 1.0:
                new_w = max(1, int(round(w * scale)))
                new_h = max(1, int(round(h * scale)))
                frame_bgr = cv2.resize(frame_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
            out_path = SOURCE_FRAMES / f'{written:05d}.jpeg'
            cv2.imwrite(str(out_path), frame_bgr)
            written += 1
        source_idx += 1
    cap.release()
else:
    cap.release()
    raw_frame_dir = Path(HOME) / f"{Path(SOURCE_VIDEO).stem}_ffmpeg_frames"
    if raw_frame_dir.exists():
        shutil.rmtree(raw_frame_dir)
    raw_frame_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['ffmpeg', '-y', '-loglevel', 'error', '-i', SOURCE_VIDEO, '-q:v', '2', str(raw_frame_dir / '%05d.jpeg')],
        check=True,
    )
    raw_paths = sorted(raw_frame_dir.glob('*.jpeg'))
    total = len(raw_paths)
    fps = float(globals().get('SOURCE_VIDEO_FPS', 0.0) or 0.0)
    stop = total if END_IDX is None else min(total, int(END_IDX))
    if start >= stop:
        raise ValueError(f'Invalid frame range: START_IDX={START_IDX}, END_IDX={END_IDX}, decoded total={total}')
    for raw_path in raw_paths[start:stop:stride]:
        frame_bgr = cv2.imread(str(raw_path), cv2.IMREAD_COLOR)
        if frame_bgr is None:
            continue
        h, w = frame_bgr.shape[:2]
        scale = min(1.0, float(TARGET_LONG_SIDE) / max(h, w)) if TARGET_LONG_SIDE else 1.0
        if scale != 1.0:
            new_w = max(1, int(round(w * scale)))
            new_h = max(1, int(round(h * scale)))
            frame_bgr = cv2.resize(frame_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(SOURCE_FRAMES / f'{written:05d}.jpeg'), frame_bgr)
        written += 1

if written == 0:
    raise RuntimeError('Frame extraction wrote zero frames.')
print(f'Extracted {written} frames to {SOURCE_FRAMES}')
print(f'   Source total frames: {total}, fps: {fps}')


In [ ]:
# Cell 11.5: Define output paths immediately after extracting frames
SOURCE_FRAME_PATHS = sorted([str(p) for p in Path(SOURCE_FRAMES).glob('*.jpeg')])
if not SOURCE_FRAME_PATHS:
    raise RuntimeError(f'No extracted frames found in {SOURCE_FRAMES}')

RUN_OUTPUT_DIR = INFERENCE_OUTPUT_ROOT / Path(SOURCE_VIDEO).stem
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INFERENCE_RESULTS = {}
TARGET_VIDEO = None
MASK_ONLY_VIDEO = None
MASK_FRAME_DIR = None
print(f'Paths defined: {len(SOURCE_FRAME_PATHS)} frames')
print('Output folder:', RUN_OUTPUT_DIR)


In [ ]:
# Cell 12: Inference state is initialized per model variant
print('Inference states will be created inside run_sam2_variant().')


In [ ]:
# Cell 13: No global predictor reset is needed
print('Each model variant is built, prompted, propagated, and released sequentially.')


In [ ]:
# Cell 14: Encode image for prompt widgets
def encode_image(filepath):
    with open(filepath, 'rb') as f:
        image_bytes = f.read()
    encoded = str(base64.b64encode(image_bytes), 'utf-8')
    return 'data:image/jpg;base64,' + encoded


In [ ]:
# Cell 14.5: Preview the first extracted frame before placing prompts
from IPython.display import Image as DisplayImage, display
print('First extracted frame for prompt placement:', SOURCE_FRAME_PATHS[0])
display(DisplayImage(filename=SOURCE_FRAME_PATHS[0]))


In [ ]:
# Cell 15: Define objects and output behavior
OBJECTS = ['particle']

# The starter defaults to the fine-tuned model. Add 'base' here if you want a
# side-by-side public-SAM2 comparison for the same prompts.
MODEL_VARIANTS_TO_RUN = ['finetuned']
PRIMARY_OUTPUT_VARIANT = 'finetuned'
MASK_REVIEW_WARN_FRACTION = 0.25
MASK_REVIEW_LARGE_FRAME_FRACTION = 0.60
DOWNLOAD_OUTPUTS = True


In [ ]:
# Cell 16: Interactive prompt points
if True:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
    import io
    from PIL import Image as PILImage

    frame_boxes = {}
    current_points = []
    FRAME_IDX = 0
    total_frames = len(SOURCE_FRAME_PATHS)

    frame_slider = widgets.IntSlider(
        value=0, min=0, max=total_frames - 1, step=1,
        description='Frame:', continuous_update=False,
        orientation='horizontal', readout=True, readout_format='d',
        layout=widgets.Layout(width='60%')
    )
    prev_button = widgets.Button(description='Previous', button_style='info', layout=widgets.Layout(width='120px'))
    next_button = widgets.Button(description='Next', button_style='info', layout=widgets.Layout(width='120px'))
    save_button = widgets.Button(description='Save Points', button_style='success', layout=widgets.Layout(width='120px'))
    clear_button = widgets.Button(description='Clear Current', button_style='warning', layout=widgets.Layout(width='120px'))
    undo_button = widgets.Button(description='Undo Last', button_style='danger', layout=widgets.Layout(width='120px'))
    status_output = widgets.Output()
    image_output = widgets.Output()
    click_instruction = widgets.HTML(value='<b>Choose foreground/background, then click the image or enter X/Y.</b>')
    point_type_toggle = widgets.ToggleButtons(
        options=[('Foreground', 1), ('Background', 0)],
        value=1,
        description='Point:',
        layout=widgets.Layout(width='260px'),
    )
    x_input = widgets.IntText(value=0, description='X:', layout=widgets.Layout(width='150px'))
    y_input = widgets.IntText(value=0, description='Y:', layout=widgets.Layout(width='150px'))
    add_point_button = widgets.Button(description='Add Point', button_style='primary', layout=widgets.Layout(width='150px'))
    bbox_widget_container = None
    _updating = False

    def _point_label(point):
        return int(point.get('point_label', point.get('prompt_label', 1)))

    def draw_circles_on_frame(frame_path, points):
        img = cv2.imread(frame_path)
        if img is None:
            raise RuntimeError(f'Could not read frame: {frame_path}')
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        for pt in points:
            x, y = int(pt['x']), int(pt['y'])
            color = (255, 0, 0) if _point_label(pt) == 1 else (0, 96, 255)
            cv2.circle(img_rgb, (x, y), 10, color, -1)
            cv2.circle(img_rgb, (x, y), 10, color, 2)
            cv2.line(img_rgb, (x - 15, y), (x + 15, y), color, 2)
            cv2.line(img_rgb, (x, y - 15), (x, y + 15), color, 2)
        return img_rgb

    def display_current_frame():
        global FRAME_IDX, current_points, bbox_widget_container, _updating
        if _updating:
            return
        _updating = True
        frame_path = SOURCE_FRAME_PATHS[FRAME_IDX]
        img_with_circles = draw_circles_on_frame(frame_path, current_points)
        temp_path = Path(HOME) / f'temp_annotated_{FRAME_IDX}.jpg'
        cv2.imwrite(str(temp_path), cv2.cvtColor(img_with_circles, cv2.COLOR_RGB2BGR))
        with image_output:
            clear_output(wait=True)
            bbox_widget_container = BBoxWidget(classes=OBJECTS)
            bbox_widget_container.image = encode_image(str(temp_path))
            bbox_widget_container.observe(on_widget_change, names='bboxes')
            display(bbox_widget_container)
            print(f'Frame {FRAME_IDX} | {len(current_points)} point(s)')
        _updating = False

    def on_widget_change(change):
        global _updating, bbox_widget_container, current_points
        if _updating:
            return
        if bbox_widget_container is not None and bbox_widget_container.bboxes:
            existing_coords = set((pt['x'], pt['y'], _point_label(pt)) for pt in current_points)
            new_points_added = False
            for wp in bbox_widget_container.bboxes:
                wx, wy = wp['x'], wp['y']
                point_kind = int(point_type_toggle.value)
                key = (wx, wy, point_kind)
                if key not in existing_coords:
                    current_points.append({'x': wx, 'y': wy, 'label': 'particle', 'point_label': point_kind})
                    new_points_added = True
                    point_name = 'foreground' if point_kind == 1 else 'background'
                    with status_output:
                        clear_output(wait=True)
                        print(f'Added {point_name} point at ({wx}, {wy}); total={len(current_points)}')
            if new_points_added:
                display_current_frame()

    def add_manual_point(b=None):
        global current_points
        x, y = x_input.value, y_input.value
        point_kind = int(point_type_toggle.value)
        current_points.append({'x': x, 'y': y, 'label': 'particle', 'point_label': point_kind})
        point_name = 'foreground' if point_kind == 1 else 'background'
        with status_output:
            clear_output(wait=True)
            print(f'Added {point_name} point at ({x}, {y}); total={len(current_points)}')
        display_current_frame()

    def save_points(b=None):
        global FRAME_IDX, current_points, frame_boxes
        if current_points:
            frame_boxes[FRAME_IDX] = current_points.copy()
            with status_output:
                clear_output(wait=True)
                print(f'Saved {len(current_points)} point(s) for frame {FRAME_IDX}; prompted frames={len(frame_boxes)}')
        else:
            with status_output:
                clear_output(wait=True)
                print(f'No points to save for frame {FRAME_IDX}')

    def clear_points(b=None):
        global current_points
        current_points = []
        display_current_frame()
        with status_output:
            clear_output(wait=True)
            print(f'Cleared points from frame {FRAME_IDX}')

    def undo_last(b=None):
        global current_points
        if current_points:
            removed = current_points.pop()
            display_current_frame()
            with status_output:
                clear_output(wait=True)
                print(f'Removed point at ({removed["x"]}, {removed["y"]}); remaining={len(current_points)}')
        else:
            with status_output:
                clear_output(wait=True)
                print(f'No points to undo on frame {FRAME_IDX}')

    def change_frame(change):
        global FRAME_IDX, current_points, frame_boxes
        if current_points:
            frame_boxes[FRAME_IDX] = current_points.copy()
        FRAME_IDX = change['new']
        current_points = frame_boxes.get(FRAME_IDX, []).copy()
        display_current_frame()
        with status_output:
            clear_output(wait=True)
            print(f'Frame {FRAME_IDX}; loaded {len(current_points)} point(s)')

    def go_prev(b):
        if frame_slider.value > 0:
            frame_slider.value -= 1

    def go_next(b):
        if frame_slider.value < total_frames - 1:
            frame_slider.value += 1

    prev_button.on_click(go_prev)
    next_button.on_click(go_next)
    save_button.on_click(save_points)
    clear_button.on_click(clear_points)
    undo_button.on_click(undo_last)
    add_point_button.on_click(add_manual_point)
    frame_slider.observe(change_frame, names='value')

    display(widgets.HBox([prev_button, next_button, save_button, clear_button, undo_button]))
    display(frame_slider)
    display(widgets.HBox([point_type_toggle, x_input, y_input, add_point_button]))
    display(status_output)
    display(click_instruction)
    display(image_output)
    display_current_frame()


In [ ]:
# Cell 16.5: Interactive bounding boxes
if True:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import cv2
    from pathlib import Path
    import numpy as np

    frame_bboxes = {}
    frames_with_points = sorted(frame_boxes.keys())
    if not frames_with_points:
        print('No frames with points. Run Cell 16 first and save at least one point prompt.')
    else:
        print(f'Found {len(frames_with_points)} frame(s) with points.')
        print('Draw one widget box. Widget boxes use top-left x/y plus width/height.')
        bbox_slider = widgets.IntSlider(
            value=0, min=0, max=len(frames_with_points) - 1, step=1,
            description='Frame:', continuous_update=False,
            layout=widgets.Layout(width='80%')
        )
        bbox_display = widgets.Output()
        propagate_button = widgets.Button(description='Propagate Relative to Points', button_style='warning', layout=widgets.Layout(width='250px'))
        bbox_status = widgets.Output()
        bbox_widget = None
        current_display_frame_idx = frames_with_points[0]

        def propagate_box_relative(button=None):
            global bbox_widget, current_display_frame_idx, frame_bboxes, frame_boxes
            if bbox_widget is None or not bbox_widget.bboxes:
                with bbox_status:
                    clear_output(wait=True)
                    print('Draw a box first.')
                return
            actual_boxes = [box for box in bbox_widget.bboxes if box.get('width', 0) > 0 and box.get('height', 0) > 0]
            if not actual_boxes:
                return
            template_box = actual_boxes[0]
            ref_point = frame_boxes[current_display_frame_idx][0]
            offset_x = template_box['x'] - ref_point['x']
            offset_y = template_box['y'] - ref_point['y']
            box_w = template_box['width']
            box_h = template_box['height']
            count = 0
            for f_idx in frames_with_points:
                if f_idx in frame_boxes and frame_boxes[f_idx]:
                    target_point = frame_boxes[f_idx][0]
                    frame_bboxes[f_idx] = [{
                        'x': target_point['x'] + offset_x,
                        'y': target_point['y'] + offset_y,
                        'width': box_w,
                        'height': box_h,
                        'label': 'particle',
                    }]
                    count += 1
            with bbox_status:
                clear_output(wait=True)
                print(f'Propagated top-left box size {int(box_w)}x{int(box_h)} across {count} frame(s).')

        def update_bbox_frame(change):
            global bbox_widget, current_display_frame_idx
            current_display_frame_idx = frames_with_points[change['new']]
            with bbox_display:
                clear_output(wait=True)
                frame_path = Path(SOURCE_FRAMES) / f'{current_display_frame_idx:05d}.jpeg'
                img = cv2.imread(str(frame_path))
                if img is None:
                    raise RuntimeError(f'Could not read prompt frame: {frame_path}')
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                for pt in frame_boxes.get(current_display_frame_idx, []):
                    color = (255, 0, 0) if int(pt.get('point_label', 1)) == 1 else (0, 96, 255)
                    cv2.circle(img, (int(pt['x']), int(pt['y'])), 8, color, -1)
                temp_path = Path(HOME) / 'temp_bbox_frame.jpg'
                cv2.imwrite(str(temp_path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
                bbox_widget = BBoxWidget(classes=OBJECTS)
                bbox_widget.image = encode_image(str(temp_path))
                display(bbox_widget)

        propagate_button.on_click(propagate_box_relative)
        bbox_slider.observe(update_bbox_frame, names='value')
        display(bbox_slider, propagate_button, bbox_status, bbox_display)
        update_bbox_frame({'new': 0})


In [ ]:
# Cell 16.75: Normalize prompt annotations and write prompt manifest
import json

if not frame_boxes:
    raise RuntimeError('No point prompts are defined.')

sample_prompt_frame = cv2.imread(SOURCE_FRAME_PATHS[0])
if sample_prompt_frame is None:
    raise RuntimeError(f'Could not read first extracted frame: {SOURCE_FRAME_PATHS[0]}')
FRAME_HEIGHT, FRAME_WIDTH = sample_prompt_frame.shape[:2]

def point_prompt_label(point):
    return int(point.get('point_label', point.get('prompt_label', 1)))

def box_to_xyxy(box, frame_width=FRAME_WIDTH, frame_height=FRAME_HEIGHT):
    if box is None:
        return None
    if 'xyxy' in box:
        x1, y1, x2, y2 = box['xyxy']
    elif all(k in box for k in ('x1', 'y1', 'x2', 'y2')):
        x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
    elif all(k in box for k in ('x', 'y', 'width', 'height')):
        # jupyter_bbox_widget uses top-left x/y with width/height.
        x1 = box['x']
        y1 = box['y']
        x2 = box['x'] + box['width']
        y2 = box['y'] + box['height']
    else:
        raise ValueError(f'Unsupported box format: {box}')
    x1 = float(np.clip(x1, 0, frame_width - 1))
    y1 = float(np.clip(y1, 0, frame_height - 1))
    x2 = float(np.clip(x2, 0, frame_width - 1))
    y2 = float(np.clip(y2, 0, frame_height - 1))
    if x2 <= x1 or y2 <= y1:
        raise ValueError(f'Invalid clipped box: {(x1, y1, x2, y2)} from {box}')
    return [x1, y1, x2, y2]

def prompt_arrays(points_list, label):
    selected = [p for p in points_list if p.get('label') == label]
    points = np.array([[p['x'], p['y']] for p in selected], dtype=np.float32)
    labels = np.array([point_prompt_label(p) for p in selected], dtype=np.int32)
    return points, labels

prompt_manifest = {
    'schema_version': 'syniscopy-sam2-prompt-v2',
    'source_video': str(SOURCE_VIDEO),
    'source_frames': str(SOURCE_FRAMES),
    'frame_count': len(SOURCE_FRAME_PATHS),
    'frame_width': int(FRAME_WIDTH),
    'frame_height': int(FRAME_HEIGHT),
    'model_variants_to_run': list(MODEL_VARIANTS_TO_RUN),
    'prompts': [],
}
if 'USE_FIXED_PROMPT' in globals():
    prompt_manifest['use_fixed_prompt'] = bool(USE_FIXED_PROMPT)

for frame_idx in sorted(set(frame_boxes.keys()) | set(frame_bboxes.keys())):
    boxes = [dict(box) for box in frame_bboxes.get(frame_idx, [])]
    prompt_manifest['prompts'].append({
        'frame_idx': int(frame_idx),
        'frame_path': str(Path(SOURCE_FRAMES) / f'{int(frame_idx):05d}.jpeg'),
        'points': [dict(point) for point in frame_boxes.get(frame_idx, [])],
        'boxes': boxes,
        'box_xyxy': [box_to_xyxy(box) for box in boxes],
    })

PROMPT_MANIFEST_PATH = RUN_OUTPUT_DIR / 'prompt_manifest.json'
with open(PROMPT_MANIFEST_PATH, 'w', encoding='utf-8') as fh:
    json.dump(prompt_manifest, fh, indent=2, sort_keys=True)

print('Prompt manifest:', PROMPT_MANIFEST_PATH)
for item in prompt_manifest['prompts']:
    pos = sum(1 for p in item['points'] if point_prompt_label(p) == 1)
    neg = sum(1 for p in item['points'] if point_prompt_label(p) == 0)
    print('  frame', item['frame_idx'], 'positive', pos, 'negative', neg, 'boxes_xyxy', item['box_xyxy'])


In [ ]:
# Cell 17: Prompt helper functions for each SAM2 predictor
def add_prompt_to_predictor(predictor, inference_state, frame_idx, object_id, points, labels, box_prompt=None):
    kwargs = dict(
        inference_state=inference_state,
        frame_idx=int(frame_idx),
        obj_id=int(object_id),
        points=points,
        labels=labels,
    )
    if box_prompt is not None:
        kwargs['box'] = box_prompt
    if hasattr(predictor, 'add_new_points_or_box'):
        return predictor.add_new_points_or_box(**kwargs)
    return predictor.add_new_points(**kwargs)

def add_all_prompts_to_predictor(predictor, inference_state):
    added = []
    for frame_idx, points_list in frame_boxes.items():
        for object_id, label in enumerate(OBJECTS, start=1):
            points, labels = prompt_arrays(points_list, label)
            if len(points) == 0:
                continue
            box_prompt = None
            if frame_idx in frame_bboxes:
                bbox_objects = [bbox for bbox in frame_bboxes[frame_idx] if bbox.get('label') == label]
                if bbox_objects:
                    box_prompt = np.array(box_to_xyxy(bbox_objects[0]), dtype=np.float32)
            try:
                add_prompt_to_predictor(predictor, inference_state, frame_idx, object_id, points, labels, box_prompt=box_prompt)
            except TypeError:
                if box_prompt is None:
                    raise
                box_points = np.array([[box_prompt[0], box_prompt[1]], [box_prompt[2], box_prompt[3]]], dtype=np.float32)
                all_points = np.vstack([points, box_points])
                all_labels = np.concatenate([labels, np.array([2, 3], dtype=np.int32)])
                add_prompt_to_predictor(predictor, inference_state, frame_idx, object_id, all_points, all_labels, box_prompt=None)
            added.append({
                'frame_idx': int(frame_idx),
                'object_id': int(object_id),
                'label': label,
                'point_count': int(len(points)),
                'positive_points': int(np.count_nonzero(labels == 1)),
                'negative_points': int(np.count_nonzero(labels == 0)),
                'box_used': box_prompt is not None,
            })
            print(
                f'Added prompt: frame={frame_idx}, object={object_id}, label={label}, '
                f'positive={np.count_nonzero(labels == 1)}, negative={np.count_nonzero(labels == 0)}, box={box_prompt is not None}'
            )
    if not added:
        raise RuntimeError('No prompts were added to the predictor.')
    return added


In [ ]:
# Cell 17.5: Display prompt points and boxes for visual checking
import matplotlib.pyplot as plt

if not PROMPT_MANIFEST_PATH.exists() or PROMPT_MANIFEST_PATH.stat().st_size == 0:
    raise FileNotFoundError(f'Prompt manifest missing: {PROMPT_MANIFEST_PATH}')

all_prompt_frames = sorted(set(frame_boxes.keys()) | set(frame_bboxes.keys()))
if not all_prompt_frames:
    print('No prompt frames found to visualize.')
else:
    num_frames = len(all_prompt_frames)
    cols = min(3, num_frames)
    rows = (num_frames + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 6 * rows))
    axes = axes.flatten() if num_frames > 1 else [axes]

    for idx, f_idx in enumerate(all_prompt_frames):
        frame_path = Path(SOURCE_FRAMES) / f'{int(f_idx):05d}.jpeg'
        frame = cv2.imread(str(frame_path))
        if frame is None:
            raise RuntimeError(f'Could not read prompt frame: {frame_path}')
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        for pt in frame_boxes.get(f_idx, []):
            x, y = int(pt['x']), int(pt['y'])
            if point_prompt_label(pt) == 1:
                cv2.circle(frame, (x, y), 5, (255, 0, 0), -1)
                cv2.drawMarker(frame, (x, y), (255, 0, 0), markerType=cv2.MARKER_CROSS, markerSize=14, thickness=1)
            else:
                cv2.circle(frame, (x, y), 4, (0, 96, 255), 1)
                cv2.drawMarker(frame, (x, y), (0, 96, 255), markerType=cv2.MARKER_TILTED_CROSS, markerSize=12, thickness=1)

        for box in frame_bboxes.get(f_idx, []):
            x1, y1, x2, y2 = [int(round(v)) for v in box_to_xyxy(box)]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)

        axes[idx].imshow(frame)
        axes[idx].set_title(f'Frame {f_idx}')
        axes[idx].axis('off')

    for i in range(num_frames, len(axes)):
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

print('Prompt manifest:', PROMPT_MANIFEST_PATH)


In [ ]:
# Cell 18: Run SAM2 inference for configured model variants
import gc
import cv2
import json
import numpy as np

sample_frame = cv2.imread(SOURCE_FRAME_PATHS[0])
if sample_frame is None:
    raise RuntimeError(f'Could not read first extracted frame: {SOURCE_FRAME_PATHS[0]}')
actual_h, actual_w = sample_frame.shape[:2]
fps_value = float(globals().get('SOURCE_VIDEO_FPS', 0.0) or 0.0)
if fps_value <= 0:
    fps_value = 5.0
video_info = sv.VideoInfo(width=actual_w, height=actual_h, fps=fps_value, total_frames=len(SOURCE_FRAME_PATHS))
print(f'VideoSink config: {actual_w}x{actual_h} @ {video_info.fps}fps')

COLORS = ['#FF1493']
mask_annotator = sv.MaskAnnotator(color=sv.ColorPalette.from_hex(COLORS), color_lookup=sv.ColorLookup.CLASS)

def variant_paths(model_variant):
    variant_dir = RUN_OUTPUT_DIR / model_variant
    mask_dir = variant_dir / 'masks'
    variant_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(SOURCE_VIDEO).stem
    return {
        'variant_dir': variant_dir,
        'overlay_video': variant_dir / f'{stem}_{model_variant}_overlay.avi',
        'mask_only_video': variant_dir / f'{stem}_{model_variant}_mask_only.avi',
        'mask_frame_dir': mask_dir,
        'metrics_json': variant_dir / 'mask_metrics.json',
    }

def _object_ids_array(object_ids):
    if hasattr(object_ids, 'detach'):
        return object_ids.detach().cpu().numpy().astype(int)
    return np.array(object_ids, dtype=int)

def run_sam2_variant(model_variant):
    print('\nRunning SAM2 variant:', model_variant)
    paths = variant_paths(model_variant)
    for old_mask in paths['mask_frame_dir'].glob('*.png'):
        old_mask.unlink()

    predictor, load_report = build_sam2_predictor_variant(model_variant)
    inference_state = predictor.init_state(video_path=SOURCE_FRAMES.as_posix())
    predictor.reset_state(inference_state)
    prompt_report = add_all_prompts_to_predictor(predictor, inference_state)

    frame_count = 0
    error_count = 0
    area_fractions = []
    per_frame = []

    with sv.VideoSink(paths['overlay_video'].as_posix(), video_info=video_info) as overlay_sink, \
         sv.VideoSink(paths['mask_only_video'].as_posix(), video_info=video_info) as mask_sink:
        for frame_idx, object_ids, mask_logits in predictor.propagate_in_video(inference_state):
            try:
                frame_path = SOURCE_FRAME_PATHS[frame_idx]
                frame = cv2.imread(str(frame_path))
                if frame is None:
                    raise RuntimeError(f'Failed to read frame {frame_idx}: {frame_path}')
                frame_h, frame_w = frame.shape[:2]
                masks = (mask_logits > 0.0).cpu().numpy().astype(bool)
                if masks.ndim == 4:
                    masks = masks.squeeze(1)
                resized_masks = np.zeros((masks.shape[0], frame_h, frame_w), dtype=bool)
                mask_frame = np.zeros((frame_h, frame_w, 3), dtype=np.uint8)
                object_ids_np = _object_ids_array(object_ids)

                frame_areas = []
                for i in range(masks.shape[0]):
                    resized_mask = cv2.resize(
                        masks[i].astype(np.uint8),
                        (frame_w, frame_h),
                        interpolation=cv2.INTER_NEAREST,
                    ).astype(bool)
                    resized_masks[i] = resized_mask
                    mask_frame[resized_mask] = [255, 255, 255]
                    object_id = int(object_ids_np[i]) if i < len(object_ids_np) else i + 1
                    cv2.imwrite(str(paths['mask_frame_dir'] / f'frame_{frame_idx:05d}_object_{object_id}.png'), resized_mask.astype(np.uint8) * 255)
                    frac = float(np.count_nonzero(resized_mask) / resized_mask.size)
                    frame_areas.append(frac)
                    area_fractions.append(frac)

                detections = sv.Detections(
                    xyxy=sv.mask_to_xyxy(masks=resized_masks),
                    mask=resized_masks,
                    class_id=object_ids_np,
                )
                annotated_frame = mask_annotator.annotate(scene=frame.copy(), detections=detections)
                overlay_sink.write_frame(annotated_frame)
                mask_sink.write_frame(mask_frame)
                frame_count += 1
                per_frame.append({
                    'frame_idx': int(frame_idx),
                    'object_ids': [int(x) for x in object_ids_np.tolist()],
                    'mask_area_fraction_max': max(frame_areas) if frame_areas else 0.0,
                    'mask_area_fraction_sum': float(sum(frame_areas)),
                })
                if frame_idx % 50 == 0:
                    print(f'   Frame {frame_idx}/{len(SOURCE_FRAME_PATHS)}')
            except Exception as exc:
                print(f'Error on frame {frame_idx}: {exc}')
                error_count += 1
                continue

    if frame_count == 0:
        raise RuntimeError(f'{model_variant} propagation wrote zero frames.')
    if not paths['overlay_video'].exists() or paths['overlay_video'].stat().st_size == 0:
        raise RuntimeError(f'Overlay video was not written: {paths["overlay_video"]}')
    if not paths['mask_only_video'].exists() or paths['mask_only_video'].stat().st_size == 0:
        raise RuntimeError(f'Mask-only video was not written: {paths["mask_only_video"]}')

    max_area = max(area_fractions) if area_fractions else 0.0
    median_area = float(np.median(area_fractions)) if area_fractions else 0.0
    review_warnings = []
    if median_area > MASK_REVIEW_WARN_FRACTION:
        review_warnings.append(f'median mask area fraction {median_area:.3f} exceeds review threshold {MASK_REVIEW_WARN_FRACTION:.3f}')
    if max_area > MASK_REVIEW_LARGE_FRAME_FRACTION:
        review_warnings.append(f'max mask area fraction {max_area:.3f} exceeds large-frame threshold {MASK_REVIEW_LARGE_FRAME_FRACTION:.3f}')

    metrics = {
        'schema_version': 'syniscopy-sam2-inference-metrics-v1',
        'model_variant': model_variant,
        'source_video': str(SOURCE_VIDEO),
        'source_frames': str(SOURCE_FRAMES),
        'frame_count_written': int(frame_count),
        'error_count': int(error_count),
        'max_mask_area_fraction': float(max_area),
        'median_mask_area_fraction': float(median_area),
        'review_warnings': review_warnings,
        'prompt_manifest': str(PROMPT_MANIFEST_PATH),
        'load_report': load_report,
        'prompt_report': prompt_report,
        'per_frame': per_frame,
    }
    with open(paths['metrics_json'], 'w', encoding='utf-8') as fh:
        json.dump(metrics, fh, indent=2)

    print(f'Completed {model_variant}: {frame_count} frames, errors={error_count}')
    print('  Overlay:', paths['overlay_video'])
    print('  Mask-only:', paths['mask_only_video'])
    print('  Masks:', paths['mask_frame_dir'])
    print('  Metrics:', paths['metrics_json'])
    if review_warnings:
        print('  Review warnings:')
        for warning in review_warnings:
            print('   -', warning)

    del predictor, inference_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    result = {k: str(v) for k, v in paths.items()}
    result['metrics'] = metrics
    return result

INFERENCE_RESULTS = {}
for variant in MODEL_VARIANTS_TO_RUN:
    INFERENCE_RESULTS[str(variant)] = run_sam2_variant(str(variant))

if PRIMARY_OUTPUT_VARIANT not in INFERENCE_RESULTS:
    PRIMARY_OUTPUT_VARIANT = next(iter(INFERENCE_RESULTS))
primary = INFERENCE_RESULTS[PRIMARY_OUTPUT_VARIANT]
TARGET_VIDEO = Path(primary['overlay_video'])
MASK_ONLY_VIDEO = Path(primary['mask_only_video'])
MASK_FRAME_DIR = Path(primary['mask_frame_dir'])
print('\nPrimary output variant:', PRIMARY_OUTPUT_VARIANT)
print('Primary overlay:', TARGET_VIDEO)
print('Primary mask-only:', MASK_ONLY_VIDEO)


In [ ]:
# Cell 19: Mask outputs are generated in Cell 18
if not INFERENCE_RESULTS:
    raise RuntimeError('No inference results found. Run Cell 18 first.')
for variant, result in INFERENCE_RESULTS.items():
    print(variant)
    print('  Overlay:', result['overlay_video'])
    print('  Mask-only:', result['mask_only_video'])
    print('  Masks:', result['mask_frame_dir'])
    print('  Metrics:', result['metrics_json'])


In [ ]:
# Cell 19.5: Report generated SAM2 output artifacts
print('Prompt manifest:', PROMPT_MANIFEST_PATH)
for variant, result in INFERENCE_RESULTS.items():
    print(f'{variant} overlay:', result['overlay_video'])
    print(f'{variant} mask-only:', result['mask_only_video'])
    print(f'{variant} per-frame masks:', result['mask_frame_dir'])
    warnings = result['metrics'].get('review_warnings', [])
    if warnings:
        print(f'{variant} review warnings:', warnings)


In [ ]:
# Cell 20: Report output paths and optionally download primary files
from google.colab import files

if TARGET_VIDEO is None or not TARGET_VIDEO.exists() or TARGET_VIDEO.stat().st_size == 0:
    raise FileNotFoundError(f'Primary overlay video missing: {TARGET_VIDEO}')
if MASK_ONLY_VIDEO is None or not MASK_ONLY_VIDEO.exists() or MASK_ONLY_VIDEO.stat().st_size == 0:
    raise FileNotFoundError(f'Primary mask-only video missing: {MASK_ONLY_VIDEO}')

print('Primary variant:', PRIMARY_OUTPUT_VARIANT)
print('Overlay video:', TARGET_VIDEO)
print('Mask-only video:', MASK_ONLY_VIDEO)
print('Per-frame masks:', MASK_FRAME_DIR)
print('All variants:', sorted(INFERENCE_RESULTS))

if DOWNLOAD_OUTPUTS:
    files.download(TARGET_VIDEO.as_posix())
    files.download(MASK_ONLY_VIDEO.as_posix())
